# Dependancies 

In [1]:
import os
from openpyxl import Workbook
import matplotlib.pyplot as plt
import pandas as pd

# Something

In [2]:
import ME_modules.MakeWriteExcel as MWE
import ME_modules.Folder_n_File_Utilities as fnfu
import  ME_modules.ME_austin_EIS.chip_processor as MCP

In [3]:
processed_chips_folder_path = None
just_do_it = False
chip_choice = None

# Main

In [ ]:
def main(use_hardcoded_path=False, hardcoded_path=None):
    # Determine master folder
    if use_hardcoded_path and hardcoded_path:
        master_folder = hardcoded_path
    else:
        master_folder = fnfu.select_folder()
        if not master_folder:
            print("No folder selected. Exiting.")
            return

    # Processed chips folder stays in same directory as main code
    try:
        base_dir = os.path.dirname(os.path.abspath(__file__))
        print(f"try base_dir: {base_dir}")
    except NameError:
        base_dir = os.getcwd()  # fallback for notebooks or interactive shells
        print(f"except base_dir: {base_dir}")

    processed_chips_folder = os.path.join(base_dir, "Processed Chips for PCAI1ia")
    os.makedirs(processed_chips_folder, exist_ok=True)

    # Output folder & log
    output_folder = os.path.join(master_folder, "Processed_Chip_Data")
    os.makedirs(output_folder, exist_ok=True)
    log_file = os.path.join(output_folder, "analysis_log.txt")
    open(log_file, 'w').close()

    # Validate chips
    chip_data = fnfu.find_valid_chips(master_folder)
    valid_chips, invalid_chips = fnfu.match_chip_files(chip_data)

    # Chip selection
    chip_choice = input("Enter chip number to process or 'all' to process all valid chips: ").strip()

    if chip_choice.lower() == 'all':
        chips_to_process = sorted(valid_chips.keys(), key=int)
        MWE.create_chips_folder_and_workbooks(processed_chips_folder)
    elif chip_choice in valid_chips:
        chips_to_process = [chip_choice]
        chip_name = f"Chip {chip_choice}"
        MWE.create_single_chip_workbook(chip_name, processed_chips_folder)
    else:
        fnfu.log_and_print(f"Invalid selection: {chip_choice}", log_file)
        return

    # 4. Processing Phase
    for chip in chips_to_process:
        chip_paths = valid_chips[chip]
        fnfu.log_and_print(f"\nProcessing Chip {chip}...", log_file)
        
        try:
            # Process the Excel file for the chip
            chip_results = MCP.process_chip_excel_only(chip, chip_paths,debug=True)

            if not chip_results:
                fnfu.log_and_print(f"No valid results for Chip {chip}", log_file)
                continue

            # Write results into correct workbook/sheets
            MCP.write_chip_results_to_workbook(chip_results, processed_chips_folder)
            fnfu.log_and_print(f"Successfully processed and wrote results for Chip {chip}", log_file)
            
        except Exception as e:
            fnfu.log_and_print(f"Error processing Chip {chip}: {str(e)}", log_file)
            continue
    
    print(f"MCP.all_raw_names_logged:{MCP.all_raw_names_logged}")
    fnfu.log_and_print("\nProcessing completed!", log_file)
    
    return os.path.abspath(processed_chips_folder) , chip_choice


In [ ]:
if __name__ == "__main__":
    processed_chips_folder_path , chip_choice = main()

In [ ]:
print("Done")

# Create Comparitive Graphs

In [ ]:
just_do_it = False
if chip_choice == "all" or just_do_it == True:
    import os
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt

    '''

    Min-max normalization (scale to 0-1):
    This method rescales the data so that the minimum value becomes 0 and the maximum becomes 1. 
    It's useful for comparing datasets with different ranges or units. The formula is: (value - min) / (max - min).

    Z-score normalization (subtract mean, divide by std):
    This method standardizes the data by subtracting the mean and dividing by the standard deviation. 
    It transforms the data into units of standard deviations from the mean, centering the distribution around zero. 
    This is helpful for identifying outliers or comparing data across different scales.

    Normalize to starting point (e.g., divide all values by the first time point value):
    This method divides all values by the first data point in the series, effectively setting the starting point to 1. 
    It's often used when studying relative changes over time. 
    This normalization highlights trends without focusing on absolute values.

    '''

    # --- 1. Path and Data Loading Setup ---
    # Get the base directory dynamically to handle both script and interactive environments
    try:
        base_dir = os.path.dirname(os.path.abspath(__file__))
    except NameError:
        base_dir = os.getcwd()

    # Define the processed chips folder path
    processed_chips_folder_path = os.path.join(base_dir, "Processed Chips for PCAI1ia")

    # Make sure the folder exists (create it if not)
    os.makedirs(processed_chips_folder_path, exist_ok=True)
    folder_path = processed_chips_folder_path

    print("Using folder:", folder_path)

    if not folder_path:
        print("No folder selected. Exiting.")
        exit()

    # Normalization options
    normalize_min_max = True
    normalize_z_score = True
    normalize_start = True
    plot_boolean = False  # set True if you want plots

    CHIP_INFO = {
        #"CHIP TEMPLATE" : ["Cas_{complex} or Cas_{only}","{0.5} or {1} or {5} Conentration of MgCl2", "{HU} protein or {SCDU} protein"],
        'Chip 21': ["Cas_complex"   ,"0.5"   ,"HU"     ],
        'Chip 22': ["Cas_complex"   ,"0.5"   ,"HU"     ],
        'Chip 23': ["Cas_complex"   ,"5"     ,"SCDU"   ],
        'Chip 26': ["Cas_complex"   ,"5"     ,"HU"     ],
        'Chip 27': ["Cas_complex"   ,"5"     ,"HU"     ],
        'Chip 32': ["Cas_complex"   ,"5"     ,"HU"     ],
        'Chip 33': ["Cas_only"      ,"1"     ,"HU"     ],
        'Chip 35': ["Cas_complex"   ,"1"     ,"HU"     ],
        'Chip 36': ["Cas_only"      ,"1"     ,"HU"     ],
        'Chip 37': ["Cas_complex"   ,"1"     ,"HU"     ],
        'Chip 39': ["Cas_complex"   ,"1"     ,"HU"     ],
        'Chip 40': ["Cas_complex"   ,"1"     ,"HU"     ],
        'Chip 41': ["Cas_complex"   ,"1"     ,"SCDU"   ],
        'Chip 43': ["Cas_complex"   ,"5"     ,"SCDU"   ],
        'Chip 44': ["Cas_complex"   ,"5"     ,"SCDU"   ],
        'Chip 45': ["Cas_only"      ,"5"     ,"HU"     ],
        'Chip 46': ["Cas_complex"   ,"1"     ,"SCDU"   ],
        'Chip 47': ["Cas_only"      ,"1"     ,"HU"     ],
        'Chip 48': ["cas_complex"   ,"5"     ,"SCDU"   ],
        'Chip 52': ["Cas_complex"   ,"5"     ,"HU"     ],
        'Chip 53': ["Cas_complex"   ,"1"     ,"SCDU"   ],
        'Chip 54': ["Cas_only"      ,"5"     ,"HU"     ],
        'Chip 55': ["Cas_complex"   ,"1"     ,"SCDU"   ],
        'Chip 56': ["Cas_only"      ,"5"     ,"HU"     ],
        'Chip 57': ["Cas_only"      ,"5"     ,"HU"     ],
        'Chip 59': ["Cas_complex"   ,"0.5"   ,"HU"     ]
    }

    # Constants
    workbook_names = [f"Chip {n}" for n in [21,22,23,26,27,32,33,35,36,37,39,40,41,43,44,45,46,47,48,52,53,54,55,56,57,59]]
    worksheet_names = ["0pM_asso", "0pM_disso", "100pM_asso", "100pM_disso", "1nM_asso", "1nM_disso", "10nM_asso", "10nM_disso", "100nM_asso", "100nM_disso"]
    headers = [
        'time(mins)', 'delta Rct-a', 'normalized Rct_a','delta Rct-d','normalized Rct_d', 'Cp1', 'Ph1', 
        'Slope 1', 'Slope 2', 'Slope 3', 'Slope 4', 'Slope 5', 
        'Angle', 'Cp_exp-a', 'Cp_exp-b', 'Ph_slope', 'Ph_peak', 
        'Area Cp', 'Area Ph', 'Area Slope', 'Area Rs-direct', 'Area Rs-Para',
        '','','linear_eq_m','linear_eq_b','Rs','delta Rct-i','Q','n'
    ]
    # Read and combine all data
    all_data = []
    for file_name in os.listdir(folder_path):
        file_path = os.path.join(folder_path, file_name)
        chip_name = os.path.splitext(file_name)[0]

        if chip_name not in workbook_names or not file_path.endswith(".xlsx"):
            continue

        xl = pd.ExcelFile(file_path)
        for sheet in xl.sheet_names:
            if sheet not in worksheet_names:
                continue

            df = xl.parse(sheet, usecols=lambda col: col in headers)
            df['chip'] = chip_name
            df['sheet'] = sheet
            all_data.append(df)

    if not all_data:
        print("No valid data found.")
        exit()

    df_all = pd.concat(all_data, ignore_index=True)

    # Add CHIP_INFO metadata
    chip_info_df = pd.DataFrame.from_dict(CHIP_INFO, orient='index', columns=['Type', 'Concentration', 'Protein']).reset_index().rename(columns={'index': 'chip'})
    df_all = df_all.merge(chip_info_df, on='chip', how='left')

    # Define plot groups
    plots = {
        "Plot 1": [("Cas_complex", "5", "HU"), ("Cas_only", "5", "HU")],
        "Plot 2": [("Cas_complex", "5", "SCDU"), ("Cas_complex", "5", "HU")],
        "Plot 3": [("Cas_complex", "0.5", "HU"), ("Cas_complex", "1", "HU"), ("Cas_complex", "5", "HU")],
        "Plot 4": [("Cas_complex", "0.5", "SCDU"), ("Cas_complex", "1", "SCDU"), ("Cas_complex", "5", "SCDU")]
    }

    y_vars = ['delta Rct-a', 'linear_eq_m']

    # Normalization methods
    normalizations = {
        'min-max': lambda y, min_val, max_val: (y - min_val) / (max_val - min_val) if (max_val - min_val) != 0 else np.nan,
        'z-score': lambda y, mean, std: (y - mean) / std if std != 0 else np.nan,
        'start': lambda y, start_val: y / start_val if start_val != 0 else np.nan
    }

    # Create main output folder
    main_output_folder = os.path.join(folder_path, "comparison graphs")
    os.makedirs(main_output_folder, exist_ok=True)

    # Create folders per normalization, per concentration, per worksheet
    for norm_name, norm_func in normalizations.items():
        norm_folder = os.path.join(main_output_folder, norm_name)
        os.makedirs(norm_folder, exist_ok=True)

        for y_var in y_vars:
            for plot_name, groups in plots.items():
                for worksheet in worksheet_names:
                    # Extract concentration prefix (e.g., '0pM' from '0pM_asso')
                    concentration = worksheet.split("_")[0]
                    condition = worksheet  # e.g., '0pM_asso'

                    # Create subfolders: concentration folder + condition folder
                    conc_folder = os.path.join(norm_folder, concentration)
                    condition_folder = os.path.join(conc_folder, condition)
                    os.makedirs(condition_folder, exist_ok=True)

                    plt.figure(figsize=(8,6))

                    for g in groups:
                        subset = df_all[
                            (df_all['Type'] == g[0]) &
                            (df_all['Concentration'] == g[1]) &
                            (df_all['Protein'] == g[2]) &
                            (df_all['sheet'] == worksheet)
                        ]

                        if subset.empty:
                            print(f"No data for {g} in {plot_name} ({worksheet})")
                            continue

                        grouped = subset.groupby('time(mins)')[y_var].mean().reset_index()
                        if grouped.empty:
                            print(f"No data for {g} in {plot_name} ({worksheet})")
                            continue

                        min_val = grouped[y_var].min()
                        max_val = grouped[y_var].max()
                        mean_val = grouped[y_var].mean()
                        std_val = grouped[y_var].std()
                        first_val = grouped[y_var].iloc[0] if not grouped.empty else 0

                        if norm_name == 'min-max':
                            norm_y = norm_func(grouped[y_var], min_val, max_val)
                        elif norm_name == 'z-score':
                            norm_y = norm_func(grouped[y_var], mean_val, std_val)
                        elif norm_name == 'start':
                            norm_y = norm_func(grouped[y_var], first_val)
                        else:
                            norm_y = grouped[y_var]

                        plt.plot(grouped['time(mins)'], norm_y, label=f"{g}")

                    plt.xlabel('time(mins)')
                    plt.ylabel(f"{y_var} ({norm_name})")
                    plt.title(f"{plot_name} - {y_var} ({worksheet}, {norm_name})")
                    plt.legend()
                    plt.tight_layout()

                    # Save figure in the appropriate subfolder
                    safe_plot_name = plot_name.replace(" ", "_")
                    safe_y_var = y_var.replace(" ", "_").replace("-", "_")
                    safe_worksheet = worksheet.replace(" ", "_").replace("-", "_")
                    filename = f"{safe_plot_name}_{safe_y_var}_{safe_worksheet}.png"
                    plt.savefig(os.path.join(condition_folder, filename))
                    plt.close()
else:
    print("Next time.")

In [ ]:
print("Done")

# Big Excel for Deepta

In [ ]:
just_do_it = True
if chip_choice == "all" or just_do_it == True:
    import os
    import pandas as pd
    import numpy as np
    from sklearn.linear_model import LinearRegression
    from scipy.optimize import curve_fit
    import matplotlib.pyplot as plt

    plot_boolean = True  # set True if you want plots

    # --- 1. Path and Data Loading Setup ---
    # Get the base directory dynamically to handle both script and interactive environments
    try:
        base_dir = os.path.dirname(os.path.abspath(__file__))
    except NameError:
        base_dir = os.getcwd()

    # Define the processed chips folder path
    processed_chips_folder_path = os.path.join(base_dir, "Processed Chips for PCAI1ia")

    # Make sure the folder exists (create it if not)
    os.makedirs(processed_chips_folder_path, exist_ok=True)
    folder_path = processed_chips_folder_path

    print("Using folder:", folder_path)

    if not folder_path:
        print("No folder selected. Exiting.")
        exit()

    # Constants
    workbook_names = [f"Chip {n}" for n in [21,22,23,26,27,32,33,35,36,37,39,40,41,43,44,45,46,47,48,52,53,54,55,56,57,59]]
    worksheet_names = [
        "0pM_asso", "0pM_disso",
        "100pM_asso", "100pM_disso",
        "1nM_asso", "1nM_disso",
        "10nM_asso", "10nM_disso",
        "100nM_asso", "100nM_disso"
    ]

    CHIP_INFO = {
        'Chip 21': ["Cas_complex", "0.5", "HU"], 'Chip 22': ["Cas_complex", "0.5", "HU"],
        'Chip 23': ["Cas_complex", "5", "SCDU"], 'Chip 26': ["Cas_complex", "5", "HU"],
        'Chip 27': ["Cas_complex", "5", "HU"], 'Chip 32': ["Cas_complex", "5", "HU"],
        'Chip 33': ["Cas_only", "1", "HU"], 'Chip 35': ["Cas_complex", "1", "HU"],
        'Chip 36': ["Cas_only", "1", "HU"], 'Chip 37': ["Cas_complex", "1", "HU"],
        'Chip 39': ["Cas_complex", "1", "HU"], 'Chip 40': ["Cas_complex", "1", "HU"],
        'Chip 41': ["Cas_complex", "1", "SCDU"], 'Chip 43': ["Cas_complex", "5", "SCDU"],
        'Chip 44': ["Cas_complex", "5", "SCDU"], 'Chip 45': ["Cas_only", "5", "HU"],
        'Chip 46': ["Cas_complex", "1", "SCDU"], 'Chip 47': ["Cas_only", "1", "HU"],
        'Chip 48': ["cas_complex", "5", "SCDU"], 'Chip 52': ["Cas_complex", "5", "HU"],
        'Chip 53': ["Cas_complex", "1", "SCDU"], 'Chip 54': ["Cas_only", "5", "HU"],
        'Chip 55': ["Cas_complex", "1", "SCDU"], 'Chip 56': ["Cas_only", "5", "HU"],
        'Chip 57': ["Cas_only", "5", "HU"], 'Chip 59': ["Cas_complex", "0.5", "HU"]
    }

    def plot_chip_results(X, y_a, y_d, slope_a, intercept_a, r2_a,
                        slope_d, intercept_d, r2_d,
                        exp_A_a, exp_b_a, r2_exp_a,
                        exp_A_d, exp_b_d, r2_exp_d,
                        chip_name, sheet, plot_boolean=False):
        
        """
        Plots delta Rct-a and delta Rct-d with both linear regression and exponential fits.
        """

        if plot_boolean == True:
            # Create figure
            plt.figure(figsize=(10, 6))

            # Raw scatter data
            plt.scatter(X, y_a, color="blue", label="ΔRct-a data")
            plt.scatter(X, y_d, color="green", label="ΔRct-d data")

            # Smooth X for curves
            x_smooth = np.linspace(min(X), max(X), 200)

            # Linear fits
            y_a_lin = slope_a * x_smooth + intercept_a
            y_d_lin = slope_d * x_smooth + intercept_d
            plt.plot(x_smooth, y_a_lin, "b--", label=f"ΔRct-a Linear (R²={r2_a:.3f})")
            plt.plot(x_smooth, y_d_lin, "g--", label=f"ΔRct-d Linear (R²={r2_d:.3f})")

            # Exponential fits (only if valid)
            if not np.isnan(exp_A_a) and not np.isnan(exp_b_a):
                y_a_exp = exp_A_a * np.exp(exp_b_a * x_smooth)
                plt.plot(x_smooth, y_a_exp, "b-", label=f"ΔRct-a Exp (R²={r2_exp_a:.3f})")

            if not np.isnan(exp_A_d) and not np.isnan(exp_b_d):
                y_d_exp = exp_A_d * np.exp(exp_b_d * x_smooth)
                plt.plot(x_smooth, y_d_exp, "g-", label=f"ΔRct-d Exp (R²={r2_exp_d:.3f})")

            # Labels and title
            plt.xlabel("Concentration")
            plt.ylabel("ΔRct")
            plt.title(f"Chip: {chip_name}, Condition: {sheet}")
            plt.legend()
            plt.grid(True, alpha=0.3)

            plt.show()
        else:
            return

    def exp_func(x, A, b):
        return A * np.exp(b * x)

    results = []

    for file_name in os.listdir(processed_chips_folder_path):
        file_path = os.path.join(processed_chips_folder_path, file_name)
        chip_name = os.path.splitext(file_name)[0]

        if chip_name not in workbook_names or not file_path.endswith(".xlsx"):
            continue

        xl = pd.ExcelFile(file_path)

        df_0_asso = xl.parse("0pM_asso") if "0pM_asso" in xl.sheet_names else None
        df_0_disso = xl.parse("0pM_disso") if "0pM_disso" in xl.sheet_names else None

        for sheet in xl.sheet_names:
            if sheet not in worksheet_names:
                continue

            df = xl.parse(sheet)
            df.columns = df.columns.str.strip()  # clean up column headers

            if df.empty:
                continue

            X = df['time(mins)'].iloc[1:].values.reshape(-1, 1)

            # --- Linear regression for delta Rct-a ---
            try:
                y_a = df['delta Rct-a'].iloc[1:].values
                model_a = LinearRegression().fit(X, y_a)
                slope_a = model_a.coef_[0]
                r2_a = model_a.score(X, y_a)
            except Exception as e:
                slope_a, r2_a, y_a, model_a = np.nan, np.nan, None, None

            # --- Linear regression for delta Rct-d ---
            try:
                y_d = df['delta Rct-d'].iloc[1:].values
                model_d = LinearRegression().fit(X, y_d)
                slope_d = model_d.coef_[0]
                r2_d = model_d.score(X, y_d)
            except Exception as e:
                slope_d, r2_d, y_d, model_d = np.nan, np.nan, None, None

            # --- Exponential fit for delta Rct-a ---
            try:
                popt_a, _ = curve_fit(exp_func, X.flatten(), y_a, maxfev=5000)
                exp_A_a, exp_b_a = popt_a
                y_pred_a = exp_func(X.flatten(), *popt_a)
                ss_res_a = np.sum((y_a - y_pred_a) ** 2)
                ss_tot_a = np.sum((y_a - np.mean(y_a)) ** 2)
                r2_exp_a = 1 - (ss_res_a / ss_tot_a) if ss_tot_a > 0 else np.nan
            except Exception:
                exp_A_a, exp_b_a, r2_exp_a, y_pred_a = np.nan, np.nan, np.nan, None

            # --- Exponential fit for delta Rct-d ---
            try:
                popt_d, _ = curve_fit(exp_func, X.flatten(), y_d, maxfev=5000)
                exp_A_d, exp_b_d = popt_d
                y_pred_d = exp_func(X.flatten(), *popt_d)
                ss_res_d = np.sum((y_d - y_pred_d) ** 2)
                ss_tot_d = np.sum((y_d - np.mean(y_d)) ** 2)
                r2_exp_d = 1 - (ss_res_d / ss_tot_d) if ss_tot_d > 0 else np.nan
            except Exception:
                exp_A_d, exp_b_d, r2_exp_d, y_pred_d = np.nan, np.nan, np.nan, None

            # Get the normalized columns
            try:
                y_norm_a = df['normalized Rct_a'].iloc[1:].values
                y_norm_d = df['normalized Rct_d'].iloc[1:].values

                # Linear regression for normalized delta Rct-a
                model_norm_a = LinearRegression().fit(X, y_norm_a)
                norm_slope_a = model_norm_a.coef_[0]
                norm_r2_a = model_norm_a.score(X, y_norm_a)

                # Linear regression for normalized delta Rct-d
                model_norm_d = LinearRegression().fit(X, y_norm_d)
                norm_slope_d = model_norm_d.coef_[0]
                norm_r2_d = model_norm_d.score(X, y_norm_d)
            except Exception:
                norm_slope_a, norm_slope_d = np.nan, np.nan
                norm_r2_a, norm_r2_d = np.nan, np.nan

            # Exponential fits
            try:
                popt_norm_a, _ = curve_fit(exp_func, X.flatten(), y_norm_a, maxfev=5000)
                norm_exp_A_a, norm_exp_b_a = popt_norm_a
                y_pred_norm_a = exp_func(X.flatten(), *popt_norm_a)
                ss_res_norm_a = np.sum((y_norm_a - y_pred_norm_a)**2)
                ss_tot_norm_a = np.sum((y_norm_a - np.mean(y_norm_a))**2)
                r2_exp_norm_a = 1 - (ss_res_norm_a / ss_tot_norm_a) if ss_tot_norm_a > 0 else np.nan
            except Exception:
                norm_exp_A_a, norm_exp_b_a, r2_exp_norm_a = np.nan, np.nan, np.nan

            try:
                popt_norm_d, _ = curve_fit(exp_func, X.flatten(), y_norm_d, maxfev=5000)
                norm_exp_A_d, norm_exp_b_d = popt_norm_d
                y_pred_norm_d = exp_func(X.flatten(), *popt_norm_d)
                ss_res_norm_d = np.sum((y_norm_d - y_pred_norm_d)**2)
                ss_tot_norm_d = np.sum((y_norm_d - np.mean(y_norm_d))**2)
                r2_exp_norm_d = 1 - (ss_res_norm_d / ss_tot_norm_d) if ss_tot_norm_d > 0 else np.nan
            except Exception:
                norm_exp_A_d, norm_exp_b_d, r2_exp_norm_d = np.nan, np.nan, np.nan



            # ✅ Call your plotting function here
            plot_chip_results(
                X.flatten(),        # X values
                y_a,                # delta Rct-a
                y_d,                # delta Rct-d
                slope_a,            # slope_a
                model_a.intercept_ if model_a else 0,   # intercept_a
                r2_a,
                slope_d,
                model_d.intercept_ if model_d else 0,   # intercept_d
                r2_d,
                exp_A_a,
                exp_b_a,
                r2_exp_a,
                exp_A_d,
                exp_b_d,
                r2_exp_d,
                chip_name,
                sheet,
                plot_boolean
            )

            # --- Subtraction vs 0pM, matched by chip and phase ---
            (
                delta_a_minus_0_slope, delta_d_minus_0_slope, 
                delta_normalized_a_minus_0_slope, delta_normalized_d_minus_0_slope,
                delta_a_minus_0_exp_A, delta_a_minus_0_exp_b,
                delta_d_minus_0_exp_A, delta_d_minus_0_exp_b,
                delta_normalized_a_minus_0_exp_A, delta_normalized_a_minus_0_exp_b,
                delta_normalized_d_minus_0_exp_A, delta_normalized_d_minus_0_exp_b
            ) = (np.nan, ) * 12
            
            chip_type, mgcl_conc, protein = CHIP_INFO.get(chip_name, ["", "", ""])
            results.append({
                "Chip Name": chip_name,
                "Type": chip_type,
                "Concentration (MgCl)": mgcl_conc,
                "Protein": protein,
                "Concentration": sheet,

                "delta Rct-a Slope": slope_a,
                "R^2_a": r2_a,
                "delta Rct-d Slope": slope_d,
                "R^2_d": r2_d,

                "delta Rct-a slope concentration minus zero concentration slope": delta_a_minus_0_slope,
                "delta Rct-d slope concentration minus zero concentration slope ": delta_d_minus_0_slope,

                "delta Rct-a Exponential A": exp_A_a,
                "delta Rct-a Exponential b": exp_b_a,
                "R^2_exp_a": r2_exp_a,
                "delta Rct-d Exponential A": exp_A_d,
                "delta Rct-d Exponential b": exp_b_d,
                "R^2_exp_d": r2_exp_d,

                "delta Rct-a exp A concentration minus zero concentration exp A": delta_a_minus_0_exp_A,
                "delta Rct-a exp b concentration minus zero concentration exp b": delta_a_minus_0_exp_b,
                "delta Rct-d exp A concentration minus zero concentration exp A ": delta_d_minus_0_exp_A,
                "delta Rct-d exp b concentration minus zero concentration exp b ": delta_d_minus_0_exp_b,

                "delta normalized Rct-a Slope": norm_slope_a,
                "R^2_a": norm_r2_a,
                "delta normalized Rct-d Slope": norm_slope_d,
                "R^2_d": norm_r2_d,

                "delta normalized Rct-a slope concentration minus zero concentration slope": delta_normalized_a_minus_0_slope,
                "delta normalized Rct-d slope concentration minus zero concentration slope": delta_normalized_d_minus_0_slope,

                "delta normalized Rct-a Exponential A": norm_exp_A_a,
                "delta normalized Rct-a Exponential b": norm_exp_b_a,
                "normalized R^2_exp_a": r2_exp_norm_a,
                "delta normalized Rct-d Exponential A": norm_exp_A_d,
                "delta normalized Rct-d Exponential b": norm_exp_b_d,
                "normalized R^2_exp_d": r2_exp_norm_d,

                "delta normalized Rct-a exp A concentration minus zero concentration exp A": delta_normalized_a_minus_0_exp_A,
                "delta normalized Rct-a exp b concentration minus zero concentration exp b": delta_normalized_a_minus_0_exp_b,
                "delta normalized Rct-d exp A concentration minus zero concentration exp A ": delta_normalized_d_minus_0_exp_A,
                "delta normalized Rct-d exp b concentration minus zero concentration exp b ": delta_normalized_d_minus_0_exp_b

            })

    # After building results list
    output_df = pd.DataFrame(results)

    # Initialize columns
    cols_to_init = [
        "delta Rct-a slope concentration minus zero concentration slope",
        "delta Rct-d slope concentration minus zero concentration slope ",
        "delta Rct-a exp A concentration minus zero concentration exp A",
        "delta Rct-a exp b concentration minus zero concentration exp b",
        "delta Rct-d exp A concentration minus zero concentration exp A ",
        "delta Rct-d exp b concentration minus zero concentration exp b ",
        "delta normalized Rct-a slope concentration minus zero concentration slope",
        "delta normalized Rct-d slope concentration minus zero concentration slope",
        "delta normalized Rct-a exp A concentration minus zero concentration exp A",
        "delta normalized Rct-a exp b concentration minus zero concentration exp b",
        "delta normalized Rct-d exp A concentration minus zero concentration exp A ",
        "delta normalized Rct-d exp b concentration minus zero concentration exp b "
    ]

    for col in cols_to_init:
        output_df[col] = np.nan

    chips = output_df['Chip Name'].unique()
    for chip in chips:
        chip_rows = output_df[output_df['Chip Name'] == chip].index

        # Reference values for this chip
        asso_ref = {
            "slope_a": output_df.loc[chip_rows[0], 'delta Rct-a Slope'],
            "slope_d": output_df.loc[chip_rows[0], 'delta Rct-d Slope'],
            "exp_A_a": output_df.loc[chip_rows[0], 'delta Rct-a Exponential A'],
            "exp_b_a": output_df.loc[chip_rows[0], 'delta Rct-a Exponential b'],
            "exp_A_d": output_df.loc[chip_rows[0], 'delta Rct-d Exponential A'],
            "exp_b_d": output_df.loc[chip_rows[0], 'delta Rct-d Exponential b'],
            "norm_slope_a": output_df.loc[chip_rows[0], 'delta normalized Rct-a Slope'],
            "norm_slope_d": output_df.loc[chip_rows[0], 'delta normalized Rct-d Slope'],
            "norm_exp_A_a": output_df.loc[chip_rows[0], 'delta normalized Rct-a Exponential A'],
            "norm_exp_b_a": output_df.loc[chip_rows[0], 'delta normalized Rct-a Exponential b'],
            "norm_exp_A_d": output_df.loc[chip_rows[0], 'delta normalized Rct-d Exponential A'],
            "norm_exp_b_d": output_df.loc[chip_rows[0], 'delta normalized Rct-d Exponential b']
        }

        disso_ref = {
            "slope_a": output_df.loc[chip_rows[1], 'delta Rct-a Slope'],
            "slope_d": output_df.loc[chip_rows[1], 'delta Rct-d Slope'],
            "exp_A_a": output_df.loc[chip_rows[1], 'delta Rct-a Exponential A'],
            "exp_b_a": output_df.loc[chip_rows[1], 'delta Rct-a Exponential b'],
            "exp_A_d": output_df.loc[chip_rows[1], 'delta Rct-d Exponential A'],
            "exp_b_d": output_df.loc[chip_rows[1], 'delta Rct-d Exponential b'],
            "norm_slope_a": output_df.loc[chip_rows[1], 'delta normalized Rct-a Slope'],
            "norm_slope_d": output_df.loc[chip_rows[1], 'delta normalized Rct-d Slope'],
            "norm_exp_A_a": output_df.loc[chip_rows[1], 'delta normalized Rct-a Exponential A'],
            "norm_exp_b_a": output_df.loc[chip_rows[1], 'delta normalized Rct-a Exponential b'],
            "norm_exp_A_d": output_df.loc[chip_rows[1], 'delta normalized Rct-d Exponential A'],
            "norm_exp_b_d": output_df.loc[chip_rows[1], 'delta normalized Rct-d Exponential b']
        }

        for i, idx in enumerate(chip_rows):
            ref = asso_ref if i % 2 == 0 else disso_ref

            # Slopes
            output_df.loc[idx, 'delta Rct-a slope concentration minus zero concentration slope'] = output_df.loc[idx, 'delta Rct-a Slope'] - ref["slope_a"]
            output_df.loc[idx, 'delta Rct-d slope concentration minus zero concentration slope '] = output_df.loc[idx, 'delta Rct-d Slope'] - ref["slope_d"]

            # Exponential
            output_df.loc[idx, 'delta Rct-a exp A concentration minus zero concentration exp A'] = output_df.loc[idx, 'delta Rct-a Exponential A'] - ref["exp_A_a"]
            output_df.loc[idx, 'delta Rct-a exp b concentration minus zero concentration exp b'] = output_df.loc[idx, 'delta Rct-a Exponential b'] - ref["exp_b_a"]
            output_df.loc[idx, 'delta Rct-d exp A concentration minus zero concentration exp A '] = output_df.loc[idx, 'delta Rct-d Exponential A'] - ref["exp_A_d"]
            output_df.loc[idx, 'delta Rct-d exp b concentration minus zero concentration exp b '] = output_df.loc[idx, 'delta Rct-d Exponential b'] - ref["exp_b_d"]

            # Normalized slopes
            output_df.loc[idx, 'delta normalized Rct-a slope concentration minus zero concentration slope'] = output_df.loc[idx, 'delta normalized Rct-a Slope'] - ref["norm_slope_a"]
            output_df.loc[idx, 'delta normalized Rct-d slope concentration minus zero concentration slope'] = output_df.loc[idx, 'delta normalized Rct-d Slope'] - ref["norm_slope_d"]

            # Normalized exponential
            output_df.loc[idx, 'delta normalized Rct-a exp A concentration minus zero concentration exp A'] = output_df.loc[idx, 'delta normalized Rct-a Exponential A'] - ref["norm_exp_A_a"]
            output_df.loc[idx, 'delta normalized Rct-a exp b concentration minus zero concentration exp b'] = output_df.loc[idx, 'delta normalized Rct-a Exponential b'] - ref["norm_exp_b_a"]
            output_df.loc[idx, 'delta normalized Rct-d exp A concentration minus zero concentration exp A '] = output_df.loc[idx, 'delta normalized Rct-d Exponential A'] - ref["norm_exp_A_d"]
            output_df.loc[idx, 'delta normalized Rct-d exp b concentration minus zero concentration exp b '] = output_df.loc[idx, 'delta normalized Rct-d Exponential b'] - ref["norm_exp_b_d"]

    # --- Save results to Excel ---
    output_file = os.path.join(folder_path, "Big Excel for Deepta.xlsx")
    output_df.to_excel(output_file, index=False)

    print(f"Results saved to: {output_file}")
else:
    print("Next time.")

In [ ]:
print("Done.")

# Big Excel file for Deependra

In [ ]:
just_do_it = True
if chip_choice == "all" or just_do_it == True:

    import os
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    from matplotlib.backends.backend_pdf import PdfPages
    from scipy.stats import linregress
    from scipy.optimize import curve_fit
    import shutil

    # --- 1. Path and Data Loading Setup ---
    try:
        base_dir = os.path.dirname(os.path.abspath(__file__))
    except NameError:
        base_dir = os.getcwd()

    processed_chips_folder_path = os.path.join(base_dir, "Processed Chips for PCAI1ia")
    os.makedirs(processed_chips_folder_path, exist_ok=True)
    folder_path = processed_chips_folder_path

    print("Using folder:", folder_path)

    if not folder_path:
        print("No folder selected. Exiting.")
        exit()

    plot_boolean = False  # set True if you want plots

    CHIP_INFO = {
        #"CHIP TEMPLATE" : ["Cas_{complex} or Cas_{only}","{0.5} or {1} or {5} Conentration of MgCl2", "{HU} protein or {SCDU} protein"],
        'Chip 21': ["Cas_complex"   ,"0.5"   ,"HU"     ],
        'Chip 22': ["Cas_complex"   ,"0.5"   ,"HU"     ],
        'Chip 23': ["Cas_complex"   ,"5"     ,"SCDU"   ],
        'Chip 26': ["Cas_complex"   ,"5"     ,"HU"     ],
        'Chip 27': ["Cas_complex"   ,"5"     ,"HU"     ],
        'Chip 32': ["Cas_complex"   ,"5"     ,"HU"     ],
        'Chip 33': ["Cas_only"      ,"1"     ,"HU"     ],
        'Chip 35': ["Cas_complex"   ,"1"     ,"HU"     ],
        'Chip 36': ["Cas_only"      ,"1"     ,"HU"     ],
        'Chip 37': ["Cas_complex"   ,"1"     ,"HU"     ],
        'Chip 39': ["Cas_complex"   ,"1"     ,"HU"     ],
        'Chip 40': ["Cas_complex"   ,"1"     ,"HU"     ],
        'Chip 41': ["Cas_complex"   ,"1"     ,"SCDU"   ],
        'Chip 43': ["Cas_complex"   ,"5"     ,"SCDU"   ],
        'Chip 44': ["Cas_complex"   ,"5"     ,"SCDU"   ],
        'Chip 45': ["Cas_only"      ,"5"     ,"HU"     ],
        'Chip 46': ["Cas_complex"   ,"1"     ,"SCDU"   ],
        'Chip 47': ["Cas_only"      ,"1"     ,"HU"     ],
        'Chip 48': ["cas_complex"   ,"5"     ,"SCDU"   ],
        'Chip 52': ["Cas_complex"   ,"5"     ,"HU"     ],
        'Chip 53': ["Cas_complex"   ,"1"     ,"SCDU"   ],
        'Chip 54': ["Cas_only"      ,"5"     ,"HU"     ],
        'Chip 55': ["Cas_complex"   ,"1"     ,"SCDU"   ],
        'Chip 56': ["Cas_only"      ,"5"     ,"HU"     ],
        'Chip 57': ["Cas_only"      ,"5"     ,"HU"     ],
        'Chip 59': ["Cas_complex"   ,"0.5"   ,"HU"     ]
    }

    # --- Constants ---
    workbook_names = [f"Chip {n}" for n in [21,22,23,26,27,32,33,35,36,37,39,40,41,43,44,45,46,47,48,52,53,54,55,56,57,59]]
    worksheet_names = ["0pM_asso", "100pM_asso", "1nM_asso", "10nM_asso", "100nM_asso",
                       "0pM_disso", "100pM_disso", "1nM_disso", "10nM_disso", "100nM_disso"]
    headers = [
        'time(mins)', 'delta Rct-a', 'normalized Rct_a','delta Rct-d','normalized Rct_d',
        'Cp1', 'Ph1','Slope 1', 'Slope 2', 'Slope 3', 'Slope 4', 'Slope 5',
        'Angle', 'Cp_exp-a', 'Cp_exp-b', 'Ph_slope', 'Ph_peak', 
        'Area Cp', 'Area Ph', 'Area Slope', 'Area Rs-direct', 'Area Rs-Para',
        '','','linear_eq_m','linear_eq_b','Rs','delta Rct-i','Q','n'
    ]
    
    # --- Collect All Data ---
    all_data = []
    for file_name in os.listdir(folder_path):
        file_path = os.path.join(folder_path, file_name)
        chip_name = os.path.splitext(file_name)[0]

        if chip_name not in workbook_names or not file_path.endswith(".xlsx"):
            continue

        xl = pd.ExcelFile(file_path)
        for sheet in xl.sheet_names:
            if sheet not in worksheet_names:
                continue
            df = xl.parse(sheet, usecols=lambda col: col in headers)
            df['chip'] = chip_name
            df['sheet'] = sheet
            all_data.append(df)

    if not all_data:
        print("No valid data found.")
        exit()

    df_all = pd.concat(all_data, ignore_index=True)

    # Add CHIP_INFO metadata
    chip_info_df = pd.DataFrame.from_dict(
        CHIP_INFO, orient='index',
        columns=['Type', 'Concentration', 'Protein']
    ).reset_index().rename(columns={'index': 'chip'})
    df_all = df_all.merge(chip_info_df, on='chip', how='left')

    # --- Define exponential fit ---
    def exp_func(x, A, b):
        return A * np.exp(b * x)

    def safe_exp_fit(x, y):
        try:
            popt, _ = curve_fit(exp_func, x, y, maxfev=5000)
            y_pred = exp_func(x, *popt)
            ss_res = np.sum((y - y_pred)**2)
            ss_tot = np.sum((y - np.mean(y))**2)
            r2 = 1 - (ss_res / ss_tot) if ss_tot > 0 else np.nan
            return popt[0], popt[1], r2
        except:
            return np.nan, np.nan, np.nan

    def plot_chip_results(x, y, slope, intercept, r2_lin,
                          exp_A, exp_b, r2_exp,
                          chip, sheet, value_col, pdf):
        plt.figure(figsize=(8, 5))
        plt.scatter(x, y, label="data", alpha=0.7)
        plt.plot(x, slope*x + intercept, "r--", label=f"Linear (R²={r2_lin:.2f})")
        if not np.isnan(exp_A) and not np.isnan(exp_b):
            plt.plot(x, exp_func(x, exp_A, exp_b), "g-", label=f"Exp (R²={r2_exp:.2f})")
        plt.xlabel("Time (mins)")
        plt.ylabel(value_col)
        plt.title(f"{chip} | {sheet}")
        plt.legend()
        plt.grid(alpha=0.3)
        pdf.savefig()
        plt.close()

    # --- Results Collector ---
    results = []

    # --- Main Processing Loop ---
    for value_col in ["normalized Rct_a", "normalized Rct_d"]:
        print(f"\nProcessing: {value_col}\n")

        # Filter
        df_mgcl1_com = df_all[(df_all['Concentration'] == "1") &
                            (df_all['Type'] == "Cas_complex")]

        # Pivot table export
        pivot_df = df_mgcl1_com.pivot_table(
            index='time(mins)',
            columns=['sheet', 'Protein', 'chip'],
            values=value_col,
            aggfunc='first'
        ).reset_index()

        # --- Save Normal Excel with multi-row headers ---
        output_file = os.path.join(folder_path, f"Big Excel {value_col}.xlsx")
        with pd.ExcelWriter(output_file, engine='xlsxwriter', engine_kwargs={'options': {'nan_inf_to_errors': True}}) as writer:
            
            # 1. Manually create the worksheet
            workbook = writer.book
            worksheet = workbook.add_worksheet("Summary")

            # 2. Get the columns
            existing_cols = pivot_df.columns.drop('time(mins)')
            existing_cols = sorted(existing_cols, key=lambda x: (
                ["0pM_asso", "100pM_asso", "1nM_asso", "10nM_asso", "100nM_asso",
                "0pM_disso", "100pM_disso", "1nM_disso", "10nM_disso", "100nM_disso"].index(x[0]),
                ["HU", "SCDU"].index(x[1]),
                int(x[2].split(' ')[1])
            ))
            
            # 3. Create the final DataFrame
            pivot_df_ordered = pivot_df[['time(mins)']].join(pivot_df[existing_cols])
            
            # 4. Write custom headers (as before)
            header_format = workbook.add_format({'bold': True, 'border': 1, 'align': 'center', 'valign': 'vcenter'})
            
            cols = pivot_df_ordered.columns

            worksheet.write(0, 0, "time(mins)")
            for j, col in enumerate(cols[1:], start=1):
                worksheet.write(0, j, col[0])
                
            worksheet.write(1, 0, "")
            for j, col in enumerate(cols[1:], start=1):
                worksheet.write(1, j, col[1])

            worksheet.write(2, 0, "")
            for j, col in enumerate(cols[1:], start=1):
                worksheet.write(2, j, col[2])
                
            # 5. Write data (as before)
            for i, row_data in enumerate(pivot_df_ordered.values.tolist()):
                # Replace NaN/Inf values with empty strings or other placeholders
                row_data = ['' if pd.isna(x) or np.isinf(x) else x for x in row_data]
                worksheet.write_row(i + 3, 0, row_data)
            
        print(f"Excel saved to: {output_file}")

        # --- PDF Plots + Results Collection ---
        def process_and_plot(group_df, pdf, label):
            for (chip, sheet), sub_df in group_df.groupby(["chip", "sheet"]):
                x = sub_df["time(mins)"].iloc[1:].values
                y = sub_df[value_col].iloc[1:].values
                if len(x) < 2:
                    continue
                slope, intercept, r_val, _, _ = linregress(x, y)
                exp_A, exp_b, r2_exp = safe_exp_fit(x, y)

                # Store results
                results.append({
                    "Chip": chip,
                    "Sheet": sheet,
                    "Protein": sub_df["Protein"].iloc[0],
                    "Value": value_col,
                    "Type": sub_df["Type"].iloc[0],
                    "Concentration": sub_df["Concentration"].iloc[0],
                    "Baseline": label,
                    "Slope": slope,
                    "Intercept": intercept,
                    "R² Linear": r_val**2,
                    "Exp A": exp_A,
                    "Exp b": exp_b,
                    "R² Exp": r2_exp,
                })

                plot_chip_results(x, y, slope, intercept, r_val**2, exp_A, exp_b, r2_exp,
                                  chip, sheet, value_col, pdf)

        # Raw
        pdf_file = os.path.join(folder_path, f"Rct_vs_time_plots_{value_col}.pdf")
        with PdfPages(pdf_file) as pdf:
            process_and_plot(df_mgcl1_com, pdf, label="raw")
        print(f"PDF saved to: {pdf_file}")

        '''
        # Subtracted
        if subtracted:
            pdf_file_sub = os.path.join(folder_path, f"Rct_vs_time_plots_{value_col}_Subtracted.pdf")
            with PdfPages(pdf_file_sub) as pdf:
                process_and_plot(df_sub, pdf, label="subtracted")
            print(f"PDF saved to: {pdf_file_sub}")
        '''

    # --- Save Results Summary ---
    results_df = pd.DataFrame(results)
    results_file = os.path.join(folder_path, "Fit_Results.xlsx")
    results_df.to_excel(results_file, index=False)
    print(f"Fit results saved to: {results_file}")

    # --- File Organization ---
    output_subfolder = "Big Excel Folder for Deependra"
    output_path = os.path.join(folder_path, output_subfolder)
    os.makedirs(output_path, exist_ok=True)
    print(f"Created new folder: {output_path}")

    files_to_move = [
        f"Big Excel normalized Rct_a.xlsx",
        f"Big Excel normalized Rct_a Subtracted.xlsx",
        f"Rct_vs_time_plots_normalized Rct_a.pdf",
        f"Rct_vs_time_plots_normalized Rct_a_Subtracted.pdf",
        f"Big Excel normalized Rct_d.xlsx",
        f"Big Excel normalized Rct_d Subtracted.xlsx",
        f"Rct_vs_time_plots_normalized Rct_d.pdf",
        f"Rct_vs_time_plots_normalized Rct_d_Subtracted.pdf",
        f"Fit_Results.xlsx"
    ]

    for filename in files_to_move:
        source_file_path = os.path.join(folder_path, filename)
        destination_file_path = os.path.join(output_path, filename)
        if os.path.exists(source_file_path):
            try:
                shutil.move(source_file_path, destination_file_path)
                print(f"Moved: {filename}")
            except Exception as e:
                print(f"Error moving {filename}: {e}")
        else:
            print(f"Skipping: {filename} (File not found)")

    print("File organization complete. ✨")
    print("Done.")

else:
    print("Next time.")

Using folder: c:\Users\austi\OneDrive - UC San Diego\Documents\AranLab\BioSensorAnalysisSrcCode\EIS_Analysis\Austin_EIS\MegaExcel\Processed Chips for PCAI1ia


C:\Users\austi\AppData\Local\Temp\ipykernel_13596\354956931.py:95: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_all = pd.concat(all_data, ignore_index=True)
C:\Users\austi\AppData\Local\Temp\ipykernel_13596\354956931.py:163: PerformanceWarning: dropping on a non-lexsorted multi-index without a level parameter may impact performance.
  existing_cols = pivot_df.columns.drop('time(mins)')



Processing: normalized Rct_a

Excel saved to: c:\Users\austi\OneDrive - UC San Diego\Documents\AranLab\BioSensorAnalysisSrcCode\EIS_Analysis\Austin_EIS\MegaExcel\Processed Chips for PCAI1ia\Big Excel normalized Rct_a.xlsx
PDF saved to: c:\Users\austi\OneDrive - UC San Diego\Documents\AranLab\BioSensorAnalysisSrcCode\EIS_Analysis\Austin_EIS\MegaExcel\Processed Chips for PCAI1ia\Rct_vs_time_plots_normalized Rct_a.pdf

Processing: normalized Rct_d

Excel saved to: c:\Users\austi\OneDrive - UC San Diego\Documents\AranLab\BioSensorAnalysisSrcCode\EIS_Analysis\Austin_EIS\MegaExcel\Processed Chips for PCAI1ia\Big Excel normalized Rct_d.xlsx


C:\Users\austi\AppData\Local\Temp\ipykernel_13596\354956931.py:163: PerformanceWarning: dropping on a non-lexsorted multi-index without a level parameter may impact performance.
  existing_cols = pivot_df.columns.drop('time(mins)')


PDF saved to: c:\Users\austi\OneDrive - UC San Diego\Documents\AranLab\BioSensorAnalysisSrcCode\EIS_Analysis\Austin_EIS\MegaExcel\Processed Chips for PCAI1ia\Rct_vs_time_plots_normalized Rct_d.pdf
Fit results saved to: c:\Users\austi\OneDrive - UC San Diego\Documents\AranLab\BioSensorAnalysisSrcCode\EIS_Analysis\Austin_EIS\MegaExcel\Processed Chips for PCAI1ia\Fit_Results.xlsx
Created new folder: c:\Users\austi\OneDrive - UC San Diego\Documents\AranLab\BioSensorAnalysisSrcCode\EIS_Analysis\Austin_EIS\MegaExcel\Processed Chips for PCAI1ia\Big Excel Folder for Deependra
Moved: Big Excel normalized Rct_a.xlsx
Skipping: Big Excel normalized Rct_a Subtracted.xlsx (File not found)
Moved: Rct_vs_time_plots_normalized Rct_a.pdf
Skipping: Rct_vs_time_plots_normalized Rct_a_Subtracted.pdf (File not found)
Moved: Big Excel normalized Rct_d.xlsx
Skipping: Big Excel normalized Rct_d Subtracted.xlsx (File not found)
Moved: Rct_vs_time_plots_normalized Rct_d.pdf
Skipping: Rct_vs_time_plots_normalized